# Temporal Decay (Rolling-Horizon Generalization) -- EDA

Visualizes the output of `run_temporal_decay_experiment`
(`thesis.experiments.temporal_decay`). The configs come straight from the
mining parameter grid: every named entry in
`configs/screening_mining_settings.yaml` (the `max_depth` x
`max_depth_attack` grid the mining EDA settled on, growth_rate / coverage /
class_weight frozen) crossed with the `--granularities` and `--models` the
run was launched with, plus a no-symbolic baseline per granularity. There
is no separate screening / shortlist step.

Models are supervised classifiers (`logreg`, `xgboost`, ...) or one-class
anomaly detectors (`iforest`, `ocsvm` -- fit on benign rows, Platt-scaled
to an attack probability). Both are frozen and walked the same way.

For each config a schema+model is mined/fit once on window 0's train split
(W_src is always the first window) and frozen, then walked forward one
window at a time to the end of the timeline. `h=0` is W_src's own held-out
test split; every horizon after that is a fully external window.

This notebook draws:
- **§1** one degradation figure -- primary metric + FPR vs. horizon, one
  curve per grid setting, at a single granularity;
- **§2** the decay-summary table (h=0 vs. the last horizon reached);
- **§3** a per-horizon feature-importance heatmap for one selected config
  (which schema features drive the model's output, and how that shifts as
  the horizon grows).

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from thesis.paths import ARTIFACTS_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Thresholds

Edit and re-run below -- nothing past this cell needs to change.

In [ ]:
SCENARIO = "cscas"
RUN_ID = None  # None -> latest run dir for SCENARIO

PRIMARY_METRIC = "auc"  # main degradation metric (drawn alongside FPR in §1)
GRANULARITY = None  # None -> finest granularity in the run (most horizons);
                    # or pin one the run used, e.g. 0.1
INCLUDE_BASELINE = True  # §1: draw the no-symbolic baseline curve too. It
                         # sits far above the symbolic curves on FPR, which
                         # can flatten them -- set False to rescale §1 to the
                         # grid settings only.

LINE_COLOR = "#4477AA"

# §3 heatmap: which (feature_set, mining_setting, granularity, model) config
# to show -- see the printed list in that section, then set this to its row
# index.
CONFIG_SELECT = 0
HEATMAP_TOP_K = 20  # schema features shown (rows), ranked by pooled mean |importance|

In [ ]:
sweep_dir = ARTIFACTS_DIR / "experiments" / "temporal_decay" / SCENARIO
# Only consider run directories -- sweep_dir also holds _derived_shortlist.csv
# (the grid x granularities x models expansion each run writes for
# provenance), and sorting unfiltered would pick that file as "latest" since
# "_" sorts after digits, then crash treating it as a directory.
run_dirs = [p for p in sweep_dir.iterdir() if p.is_dir()]
run_dir = sweep_dir / RUN_ID if RUN_ID else sorted(run_dirs)[-1]

per_horizon = pd.read_csv(run_dir / "per_horizon_results.csv")
decay_summary = pd.read_csv(run_dir / "decay_summary.csv")
explanations = pd.read_csv(run_dir / "explanations.csv")
lime_fidelity = pd.read_csv(run_dir / "lime_fidelity.csv")

print(f"Loaded from {run_dir}")
print(f"per_horizon_results: {len(per_horizon)} rows")
print(f"decay_summary:       {len(decay_summary)} rows")
print(f"explanations:         {len(explanations)} rows")
print(f"lime_fidelity:        {len(lime_fidelity)} rows")
print(f"grid settings: {sorted(per_horizon['mining_setting'].dropna().unique())}")
print(f"granularities:  {sorted(per_horizon['granularity'].dropna().unique())}")
per_horizon.head()

## 1. Degradation curve

**One figure**, one panel per metric (`PRIMARY_METRIC` + FPR), **one curve
per grid setting**, all at a single granularity (`GRANULARITY` above --
default the finest, which has the most horizons). Each curve walks forward
from window 0 to the end of the timeline. The `h=0` point (W_src's own
held-out test split, never seen by mining or fitting) is starred; every
later point is a fully external window.

The no-symbolic baseline is the dashed black line (`INCLUDE_BASELINE`). On
FPR it sits far above the symbolic curves, so those can look flat against
it -- set `INCLUDE_BASELINE = False` to rescale to the grid settings only.

Horizon is a window index and window count scales with granularity, so one
figure fixes one granularity to keep the x-axis comparable across settings
-- change `GRANULARITY` to see another.

In [ ]:
from matplotlib.lines import Line2D

CONFIG_COLS = ["feature_set", "mining_setting", "granularity", "model"]
GROUP_COLS = [c for c in CONFIG_COLS if c != "granularity"]


def config_label(row) -> str:
    return f"{row['feature_set']}/{row['mining_setting']} g={row['granularity']:g}"


def group_label(row) -> str:
    """Short legend label for one grid setting (granularity-independent).
    The mining-grid entries share a frozen growth_rate prefix (gr3_ -- see
    the mining EDA), so drop it: gr3_md1_mda2 -> md1/mda2. Baseline carries
    no mining_setting."""
    ms = row["mining_setting"]
    if pd.isna(ms):
        return "baseline (no symbolic)"
    ms = str(ms)
    if ms.startswith("gr3_"):
        ms = ms[4:]
    return ms.replace("_", "/")


def _resolve_gran() -> float:
    grans = sorted(per_horizon["granularity"].dropna().unique())
    if GRANULARITY is None:
        return grans[0]  # finest -> most horizons -> longest decay curve
    if GRANULARITY not in grans:
        raise ValueError(f"GRANULARITY={GRANULARITY} not in this run ({grans})")
    return GRANULARITY


def _grid_settings(df: pd.DataFrame) -> list[dict]:
    """One dict per grid setting present in `df`, baseline first then mining
    settings by name. groupby(dropna=False) so the baseline row (NaN
    mining_setting) isn't dropped the way an `== NaN` mask would drop it."""
    keys = list(df.groupby(GROUP_COLS, dropna=False, sort=False).groups)
    keys.sort(key=lambda k: (k[0] != "baseline", "" if pd.isna(k[1]) else str(k[1])))
    return [dict(zip(GROUP_COLS, k)) for k in keys]


def _setting_mask(df: pd.DataFrame, s: dict) -> pd.Series:
    m = df["feature_set"].eq(s["feature_set"]) & df["model"].eq(s["model"])
    ms = s["mining_setting"]
    return m & (df["mining_setting"].isna() if pd.isna(ms) else df["mining_setting"].eq(ms))


def plot_degradation(metrics=(PRIMARY_METRIC, "fpr"), include_baseline=None) -> None:
    """One figure: each metric vs. horizon at a single granularity, one line
    per grid setting. h=0 (W_src's own held-out test split) is starred; the
    baseline (no symbolic features), if drawn, is a dashed black line.

    A shared x-axis (window index) is why this fixes one granularity: finer
    granularity -> more windows -> a longer curve, not comparable on the
    same axis."""
    include_baseline = INCLUDE_BASELINE if include_baseline is None else include_baseline
    gran = _resolve_gran()
    df = per_horizon[per_horizon["granularity"] == gran]
    settings = _grid_settings(df)
    if not include_baseline:
        settings = [s for s in settings if not pd.isna(s["mining_setting"])]

    palette = plt.get_cmap("tab10")
    mining_names = [s["mining_setting"] for s in settings if not pd.isna(s["mining_setting"])]
    color_of = {name: palette(i % 10) for i, name in enumerate(mining_names)}

    fig, axes = plt.subplots(1, len(metrics), figsize=(6.4 * len(metrics), 4.6), squeeze=False)
    for ax, metric in zip(axes[0], metrics):
        for s in settings:
            sub = df[_setting_mask(df, s)].sort_values("horizon_window_index")
            if sub.empty or metric not in sub.columns:
                continue
            baseline = pd.isna(s["mining_setting"])
            color = "black" if baseline else color_of[s["mining_setting"]]
            ax.plot(
                sub["horizon_window_index"], sub[metric],
                linestyle="--" if baseline else "-", color=color,
                marker="o", markersize=4, linewidth=1.7,
                label=group_label(s), zorder=3 if baseline else 2,
            )
            src = sub[sub["is_source_window"]]
            ax.scatter(
                src["horizon_window_index"], src[metric], color=color,
                marker="*", s=130, edgecolor="black", linewidth=0.5, zorder=5,
            )
        ax.set_xlabel("horizon (window index)")
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.grid(alpha=0.25)
        ax.margins(x=0.02)

    star = Line2D(
        [], [], color="grey", marker="*", markersize=11, linestyle="None",
        markeredgecolor="black", markeredgewidth=0.5,
    )
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(
        handles + [star], labels + ["W_src held-out (h=0)"],
        loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8,
        title=f"grid setting  (gran={gran:g})", title_fontsize=8,
    )
    fig.suptitle(
        f"{SCENARIO}: temporal decay -- one curve per grid setting (gran={gran:g})"
    )
    fig.tight_layout()


plot_degradation()

### Near-duplicate grid settings check

Two grid settings can trace the same curve above for a real reason, not a
plotting bug: AUC is a rank statistic, so if a deeper `max_depth_attack`
only adds a *weak*, marginal leaf-rule predicate that a regularized logistic
regression shrinks toward irrelevance, the model's ranking of alert groups
-- and therefore its AUC -- barely moves, even though the mined schemas
differ (different feature counts, different cache fingerprints). This cell
checks the raw numbers directly: any two settings whose per-horizon metric
values match within `atol` at every shared horizon are flagged. A non-empty
result is a signal worth taking back to the mining grid
(`configs/screening_mining_settings.yaml`) -- that `max_depth` /
`max_depth_attack` pair may not be doing meaningful work for this scenario.

In [ ]:
def find_near_duplicate_configs(df: pd.DataFrame, metric: str, atol: float = 1e-6) -> pd.DataFrame:
    pivot = df.pivot_table(index="horizon_window_index", columns=CONFIG_COLS, values=metric)
    configs = list(pivot.columns)
    rows = []
    for i in range(len(configs)):
        for j in range(i + 1, len(configs)):
            a, b = pivot[configs[i]], pivot[configs[j]]
            shared = a.notna() & b.notna()
            if shared.sum() == 0:
                continue
            diff = (a[shared] - b[shared]).abs()
            if diff.max() <= atol:
                rows.append({
                    "config_a": config_label(dict(zip(CONFIG_COLS, configs[i]))),
                    "config_b": config_label(dict(zip(CONFIG_COLS, configs[j]))),
                    "n_shared_horizons": int(shared.sum()),
                    "max_abs_diff": float(diff.max()),
                })
    return pd.DataFrame(rows)


near_dupes = find_near_duplicate_configs(per_horizon, PRIMARY_METRIC, atol=1e-6)
if near_dupes.empty:
    print(f"No config pairs with identical {PRIMARY_METRIC} (within 1e-6) at every shared horizon.")
else:
    print(
        f"{len(near_dupes)} config pair(s) with near-identical {PRIMARY_METRIC} at every "
        "shared horizon -- see the markdown note above before assuming this is a plotting bug:"
    )
near_dupes

## 2. Decay summary

Score/FPR at `h=0` (W_src's own held-out test split) vs. the last horizon
actually reached (`h_max`, the final window of the timeline), and their
difference (`decay_rate_{metric} = score(h=0) - score(h_max)`;
`fpr_drift = fpr(h_max) - fpr(h=0)`).

In [ ]:
sort_col = f"decay_rate_{PRIMARY_METRIC}"
decay_summary.sort_values(sort_col, ascending=False) if sort_col in decay_summary.columns else decay_summary

## 3. Per-horizon feature-importance heatmap

For one selected config (`CONFIG_SELECT`), a heatmap of **signed** mean
importance:

- **rows** = the top-`HEATMAP_TOP_K` schema features, ranked by pooled mean
  |importance| across all horizons;
- **columns** = horizon (window index);
- **cell** = mean signed importance over that horizon's explained sample --
  **red pushes the model's prediction toward `attack`, blue toward
  `benign`**, intensity = strength.

One panel per method (SHAP, LIME) that the run logged. Read a row
left-to-right: a feature whose colour fades or flips as the horizon grows
is one whose influence on the model's output is drifting. A grey cell means
that feature wasn't among the logged importances at that horizon (only
possible on older runs -- the experiment now logs every feature).

This shows the model's **output** attribution -- which features move the
score and which way -- not its **accuracy**. A strong row means the model
leans on that feature, not that the feature makes it correct.

In [ ]:
# Configs that have logged importances. Symbolic (schema) configs first --
# the heatmap is about the mined schema; baseline is base-features-only.
# Pick one with CONFIG_SELECT above.
available_configs = (
    explanations[CONFIG_COLS].drop_duplicates()
    .assign(_kind=lambda d: d["feature_set"].eq("baseline").astype(int))
    .sort_values(["_kind", "mining_setting", "granularity", "model"], na_position="last")
    .drop(columns="_kind")
    .reset_index(drop=True)
)


def _config_mask(df: pd.DataFrame, row) -> pd.Series:
    """Row-equality mask that treats NaN == NaN as a match (plain `==`
    never matches NaN, so baseline configs would select nothing)."""
    m = pd.Series(True, index=df.index)
    for c in CONFIG_COLS:
        v = row[c]
        m &= df[c].isna() if pd.isna(v) else df[c].eq(v)
    return m


available_configs

In [ ]:
selected = available_configs.iloc[CONFIG_SELECT]
selected_explanations = explanations[_config_mask(explanations, selected)]
print(f"selected: {config_label(selected)}  ({len(selected_explanations)} importance rows)")


def plot_importance_heatmap(sub: pd.DataFrame, title: str, k: int = HEATMAP_TOP_K) -> None:
    """Signed mean importance per (feature, horizon), top-k features by
    pooled mean |importance| across horizons (a feature absent at a horizon
    counts as 0 for ranking, so the rows shown are the consistently
    influential ones). One panel per method -- SHAP and LIME are different
    units, each ranked and colour-scaled on its own. Red pushes the
    prediction toward attack, blue toward benign; grey = feature not among
    the logged importances at that horizon (pre-"log every feature" runs)."""
    if sub.empty:
        print("No logged importances for this config -- pick another CONFIG_SELECT.")
        return
    methods = sorted(sub["method"].unique())
    horizons = sorted(sub["horizon_window_index"].unique())
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad("#e8e8e8")

    fig, axes = plt.subplots(
        1, len(methods),
        figsize=(max(4.5, 0.55 * len(horizons) + 3.0) * len(methods), 0.30 * k + 1.8),
        squeeze=False, sharey=False,
    )
    for ax, method in zip(axes[0], methods):
        pivot = sub[sub["method"] == method].pivot_table(
            index="feature", columns="horizon_window_index",
            values="importance", aggfunc="mean",
        )
        order = pivot.abs().fillna(0).mean(axis=1).sort_values(ascending=False).head(k).index
        pivot = pivot.loc[order]
        vmax = float(np.nanmax(np.abs(pivot.values))) or 1e-9

        im = ax.imshow(
            np.ma.masked_invalid(pivot.values), aspect="auto", cmap=cmap,
            norm=mcolors.TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax),
        )
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"h{c}" for c in pivot.columns], fontsize=7)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=7)
        ax.set_xlabel("horizon (window index)")
        ax.set_title(f"{method.upper()}  (red → attack, blue → benign)", fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="mean signed importance")

    fig.suptitle(title)
    fig.tight_layout()


plot_importance_heatmap(selected_explanations, config_label(selected))